# COSMoS_BC_DSS_Flatten -- COSMoS Biomedical Concepts and Dataset Specializations

**Purpose:** Build a flat interim reference file from CDISC COSMoS source files.
Flattens the two-level COSMoS graph (Biomedical Concepts → Dataset Specializations)
into a single tabular structure covering all SDTM domains.

**Output:** `interim/COSMoS_BC_DSS.xlsx`

**Pipeline:**
- `COSMoS_BC_DSS_Flatten.ipynb` (this notebook) -- downloads, flattens, exports interim file
- `COSMoS_BC_DSS_Validate.ipynb` -- reads interim file, runs QC-01 to QC-13, writes QC report


### COSMoS Source Names vs. Our Output Columns

COSMoS uses implementation-oriented field names optimised for SDTM variable
assignment. This notebook translates them to semantically transparent names
suited for study design and LLM consumption.

| COSMoS field | Our output column | Rationale |
|---|---|---|
| `bc_id` | `COSMoS_BC_ID` | COSMoS internal identifier -- may be a placeholder, not always an NCIt code |
| `ncit_code` | `NCIt_Code` | The actual NCIt C-code; blank = no NCIt assignment yet; matches SDTM_Test_Identity.xlsx join key |
| `short_name` (BC) | `BC_Name` | "label" and "short_name" are overloaded across BC and DS levels |
| `definition` | `BC_Definition` | Plain-language definition -- for LLM disambiguation, parallel to NCIt_Definition in SDTM_Test_Identity.xlsx |
| `synonyms` | `BC_Synonyms` | Qualify level for disambiguation |
| `vlm_group_id` | `DS_Code` | "VLM group" is opaque; this is the DS code |
| `testCode` | `TESTCD` | Match SDTM_Test_Identity.xlsx TESTCD column exactly |
| `testName` | `TEST` | Match SDTM_Test_Identity.xlsx TEST column |
| `assigned_term` (on Topic row) | `TESTCD_NCIt` | NCIt code for the TESTCD. Usually equals `NCIt_Code`; differs for ~6 legacy tests |
| (derived) | `BC_Scope` | Subject or Study — derived from bc_categories containing "Trial Summary" or "Clinical Trial Attribute" |
| `specimenIdentity` | `Specimen` | "Identity" is COSMoS model terminology, not plain English |
| `methodIdentity` | `Method` | Same |
| `resultScale` / `DS_Type` | `Result_Scale` | Merge two representations of the same concept |
| `Domains_with_DS` | `Domains_with_DS` | Kept as-is from COSMoS source |
| `dsCategory` / `dsSubcategory` | `COSMoS_Category` / `COSMoS_Subcategory` | Prefix signals COSMoS provenance |
| `dec_n` | `Decimal_Places` | Abbreviation is opaque |

**Columns that do not change:** `Categories`, `Hierarchy_Path`, `Parent_Label`,
`Domain_Class`, `location`, `laterality`, `evaluator` -- names are already clear.

**Dropped from original output:** `Hierarchy_Level` (redundant with path depth),
`DS_Count` (derivable from Row_Type=DS filter).


## 1. Configuration


In [ ]:
import pandas as pd
from pathlib import Path
import re
import requests
from collections import defaultdict, Counter
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter

# 
#   CACHE SWITCH                                           
#   True  = use cached files in /downloads if present      
#   False = always download fresh from COSMoS              
# 
USE_CACHE = True

# Directory structure - notebook is in notebooks/ subfolder
BASE_DIR = Path.cwd().parent
DOWNLOADS_DIR = BASE_DIR / 'downloads'
INTERIM_DIR   = BASE_DIR / 'interim'

DOWNLOADS_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# COSMoS download URLs
BC_URL = 'https://cdisc-org.github.io/COSMoS/export/cdisc_biomedical_concepts_latest.xlsx'
DS_URL = 'https://cdisc-org.github.io/COSMoS/export/cdisc_sdtm_dataset_specializations_latest.xlsx'

BC_FILE = DOWNLOADS_DIR / 'cdisc_biomedical_concepts_latest.xlsx'
DS_FILE = DOWNLOADS_DIR / 'cdisc_sdtm_dataset_specializations_latest.xlsx'

# SDTM Terminology (auto-downloaded from NCI EVS)
SDTM_CT_URL = 'https://evs.nci.nih.gov/ftp1/CDISC/SDTM/SDTM%20Terminology.txt'
SDTM_CT_FILE = DOWNLOADS_DIR / 'SDTM_Terminology.txt'

GENERATION_DATE = datetime.now().strftime('%Y-%m-%d')
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print('Configuration:')
print(f'  Cache:  {USE_CACHE}')
print(f'  BC:     {BC_FILE}')
print(f'  DS:     {DS_FILE}')
print(f'  CT:     {SDTM_CT_FILE}')
print(f'  Interim:  {INTERIM_DIR}')


## 1b. SDTM Domain Class Classification

Static classification of SDTM domains by observation class.
Used to annotate BC_DS and Domain_Summary.

| Domain Class | SDTM Meaning | Downstream Relevance |
|---|---|---|
| Findings | Measurements with controlled TESTCD | Panel composition, specimen matching |
| Events | Occurrences with free-text terms (AETERM, DSTERM) | Not panel-relevant |
| Interventions | Administered agents (CMTRT, EXTRT) | Not panel-relevant |
| Special-Purpose | Demographics, Comments, Subject Elements | Not panel-relevant |

Only **Findings** domains have controlled test code codelists, which is why
panel coverage verification (notebook 05) is architecturally scoped to Findings.


In [ ]:
# SDTM Domain Class -- hardcoded classification
# Source: SDTM CT definitions (SDTM Domain Abbreviation codelist)
# Verified against SDTM_v2.0.xlsx model metadata (not publicly downloadable).
# 24 newer/less common domains omitted -- only domains with COSMoS BC coverage
# are needed here. Trial Design lumped into Special-Purpose (intentional).
#
# Findings: quantitative/qualitative measurements with controlled test codes
# Events:   occurrences with free-text terms (AE, DS, MH, etc.)
# Interventions: administered agents/treatments (CM, EX, SU, etc.)
# Special-Purpose: demographics, comments, subject elements, etc.

DOMAIN_CLASS = {
    # Findings
    'BS': 'Findings', 'CP': 'Findings', 'CV': 'Findings',
    'DA': 'Findings', 'DD': 'Findings', 'EG': 'Findings',
    'FT': 'Findings', 'GF': 'Findings', 'IE': 'Findings',
    'IS': 'Findings', 'LB': 'Findings', 'MB': 'Findings',
    'MI': 'Findings', 'MK': 'Findings', 'MS': 'Findings',
    'NV': 'Findings', 'OE': 'Findings',
    'PC': 'Findings', 'PE': 'Findings', 'PP': 'Findings',
    'QS': 'Findings', 'RE': 'Findings', 'RP': 'Findings',
    'RS': 'Findings', 'SC': 'Findings', 'SS': 'Findings',
    'TR': 'Findings', 'TU': 'Findings', 'UR': 'Findings',
    'VS': 'Findings', 'SR': 'Findings',
    # Events
    'AE': 'Events', 'BE': 'Events', 'CE': 'Events',
    'DS': 'Events', 'DV': 'Events', 'HO': 'Events',
    'MH': 'Events',
    # Interventions
    'AG': 'Interventions', 'CM': 'Interventions', 'EC': 'Interventions',
    'EX': 'Interventions', 'ML': 'Interventions', 'PR': 'Interventions',
    'SU': 'Interventions',
    # Special-Purpose
    'CO': 'Special-Purpose', 'DM': 'Special-Purpose', 'SE': 'Special-Purpose',
    'OI': 'Special-Purpose',
    'SM': 'Special-Purpose', 'SV': 'Special-Purpose', 'TA': 'Special-Purpose',
    'TD': 'Special-Purpose', 'TE': 'Special-Purpose', 'TI': 'Special-Purpose',
    'TM': 'Special-Purpose', 'TS': 'Special-Purpose', 'TV': 'Special-Purpose',
    # Findings About (FA* domains)
    'FA': 'Findings',
}

def get_domain_class(domain):
    """Return SDTM domain class. Unknown domains flagged explicitly."""
    return DOMAIN_CLASS.get(domain, 'Unknown')

# SDTM Domain Labels -- human-readable names
# Source: SDTM CT codelist display names (via SDTM_CT_Extract notebook)
DOMAIN_LABEL = {
    'AE': 'Adverse Events', 'AG': 'Procedure Agents',
    'BE': 'Biospecimen Events', 'BS': 'Biospecimen Characteristics',
    'CE': 'Clinical Events', 'CM': 'Concomitant Medications',
    'CO': 'Comments', 'CP': 'Cell Phenotyping',
    'CV': 'Cardiovascular', 'DA': 'Drug Accountability',
    'DD': 'Death Details', 'DM': 'Demographics',
    'DS': 'Disposition', 'DV': 'Protocol Deviations',
    'EC': 'Exposure as Collected', 'EG': 'ECG Test Results',
    'EX': 'Exposure', 'FA': 'Findings About',
    'FT': 'Functional Tests', 'GF': 'Genomics Findings',
    'HO': 'Healthcare Encounters', 'IE': 'Inclusion/Exclusion',
    'IS': 'Immunogenicity Specimen', 'LB': 'Laboratory Test Results',
    'MB': 'Microbiology Specimen', 'MH': 'Medical History',
    'MI': 'Microscopic Findings', 'MK': 'Musculoskeletal',
    'ML': 'Meals', 'MS': 'Microbiology Susceptibility',
    'NV': 'Nervous System', 'OE': 'Ophthalmic Examinations',
    'OI': 'Non-host Organism Identifiers', 'PC': 'Pharmacokinetics Concentrations',
    'PE': 'Physical Examination', 'PP': 'Pharmacokinetics Parameters',
    'PR': 'Procedures', 'QS': 'Questionnaires',
    'RE': 'Respiratory', 'RP': 'Reproductive System',
    'RS': 'Disease Response', 'SC': 'Subject Characteristics',
    'SE': 'Subject Elements', 'SM': 'Subject Milestones',
    'SR': 'Skin Response', 'SS': 'Subject Status',
    'SU': 'Substance Use', 'SV': 'Subject Visits',
    'TA': 'Trial Arms', 'TD': 'Trial Disease Assessments',
    'TE': 'Trial Elements', 'TI': 'Trial Inclusion/Exclusion',
    'TM': 'Trial Disease Milestones', 'TR': 'Tumor/Lesion Results',
    'TS': 'Trial Summary', 'TU': 'Tumor/Lesion Identification',
    'TV': 'Trial Visits', 'UR': 'Urinary System',
    'VS': 'Vital Signs',
}

def get_domain_label(domain):
    """Return SDTM domain label. Unknown domains return domain code."""
    return DOMAIN_LABEL.get(domain, domain)

print(f'Domain class definitions: {len(DOMAIN_CLASS)} domains')
from collections import Counter as _Counter
_cls_counts = _Counter(DOMAIN_CLASS.values())
for cls, n in sorted(_cls_counts.items()):
    print(f'  {cls:20} {n:>3} domains')


## 2. Download COSMoS Files (Smart Caching)


In [ ]:
def download_if_needed(url, target_path, use_cache=True):
    """Download file if not cached or cache disabled."""
    if use_cache and target_path.exists():
        size = target_path.stat().st_size
        print(f'  [CACHED] {target_path.name} ({size:,} bytes)')
        return True

    print(f'  [DOWNLOADING] {url}')
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    target_path.write_bytes(response.content)
    size = target_path.stat().st_size
    print(f'  [OK] {target_path.name} ({size:,} bytes)')
    return True

print('COSMoS files:')
download_if_needed(BC_URL, BC_FILE, USE_CACHE)
download_if_needed(DS_URL, DS_FILE, USE_CACHE)

print(f'\nSDTM CT:')
download_if_needed(SDTM_CT_URL, SDTM_CT_FILE, USE_CACHE)


## 3. Load SDTM Controlled Terminology

Build lookup indexes for Specimen Type, Method, and Unit codelists.
These map DS assigned_term values to CDISC submission values and NCIt codes.


In [ ]:
ct_df = pd.read_csv(SDTM_CT_FILE, sep='\t', dtype=str, encoding='utf-8')
print(f'CT rows: {len(ct_df)}')

COL_MAP = {
    'Code': 'code', 'Codelist Code': 'codelist_code',
    'CDISC Submission Value': 'submission_value',
    'CDISC Synonym(s)': 'synonyms', 'NCI Preferred Term': 'nci_term',
}
for old, new in COL_MAP.items():
    if old in ct_df.columns:
        ct_df = ct_df.rename(columns={old: new})

print(f'Columns: {ct_df.columns.tolist()}')

In [ ]:
SPECTYPE_CODE = 'C78734'
METHOD_CODE = 'C85492'
UNIT_CODE = 'C71620'

def build_codelist_index(df, codelist_code):
    """Build lookup index for a codelist: NCIt code and submission value as keys."""
    cl_df = df[df['codelist_code'] == codelist_code].copy()
    index = {}
    for _, row in cl_df.iterrows():
        code = row.get('code', '')
        submission = row.get('submission_value', '')
        synonyms = row.get('synonyms', '')
        nci_term = row.get('nci_term', '')

        if pd.isna(code) or pd.isna(submission):
            continue

        code, submission = str(code).strip(), str(submission).strip()

        entry = {
            'ncit_code': code,
            'submission_value': submission,
            'nci_term': str(nci_term) if pd.notna(nci_term) else submission,
        }

        index[submission.upper()] = entry
        index[code] = entry

        if pd.notna(synonyms):
            for syn in str(synonyms).split(';'):
                syn = syn.strip().upper()
                if syn and syn not in index:
                    index[syn] = entry
    return index

SPECIMEN_INDEX = build_codelist_index(ct_df, SPECTYPE_CODE)
METHOD_INDEX = build_codelist_index(ct_df, METHOD_CODE)
UNIT_INDEX = build_codelist_index(ct_df, UNIT_CODE)

print(f'SPECTYPE (C78734): {len(SPECIMEN_INDEX)} entries')
print(f'METHOD (C85492):   {len(METHOD_INDEX)} entries')
print(f'UNIT (C71620):     {len(UNIT_INDEX)} entries')


## 4. Data Corrections

Known inconsistencies in COSMoS export that need correction.


In [ ]:
# Data integrity principle: COSMoS source data is shown as-is.
# Known source discrepancies are documented in QC-07 (below) for SME review,
# not silently corrected here. Corrections in the pipeline obscure the gap
# and prevent the issue from being reported back to CDISC COSMoS curators.
#
# Known issue (not corrected):
#   GLUCSERPL specifies SERUM (C13325) -- all other *SERPL DS use SERUM OR PLASMA (C105706)
#   This appears to be a COSMoS source error. Flagged in QC-07.

DS_SPECIMEN_CORRECTIONS = {}   # intentionally empty -- see principle above

print('Data corrections: none applied (data integrity principle)')


## 5. Codelist Lookup Functions


In [ ]:
# Quality tracking
UNMAPPED_SPECIMENS = {}
UNMAPPED_METHODS = {}
UNMAPPED_UNITS = {}
MAPPED_SPECIMENS = set()
MAPPED_METHODS = set()
MAPPED_UNITS = set()

def ct_lookup(value, index, tracker_mapped, tracker_unmapped, bc_id=None, vlm_id=None):
    """Generic CT lookup with quality tracking."""
    if not value:
        return None, None
    normalized = str(value).upper().strip()
    entry = index.get(normalized)
    if entry:
        tracker_mapped.add(entry['submission_value'])
        return entry['submission_value'], entry['ncit_code']
    else:
        if normalized not in tracker_unmapped:
            tracker_unmapped[normalized] = []
        tracker_unmapped[normalized].append((bc_id, vlm_id))
        return None, None


def ct_lookup_multi(value, index, tracker_mapped, tracker_unmapped, bc_id=None, vlm_id=None):
    """CT lookup for semicolon-separated value_list fields.

    Splits on semicolon, looks up each value individually, and rejoins.
    Same pattern as unit handling -- applied to specimen and method value_lists.

    Returns:
        (resolved_string, ncit_string) -- semicolon-separated resolved values
    """
    if not value:
        return None, None
    parts = [p.strip() for p in str(value).split(';') if p.strip()]
    if len(parts) <= 1:
        # Single value -- use standard lookup
        return ct_lookup(value, index, tracker_mapped, tracker_unmapped, bc_id, vlm_id)
    # Multi-value -- look up each
    resolved = []
    ncit_codes = []
    for part in parts:
        sub, ncit = ct_lookup(part, index, tracker_mapped, tracker_unmapped, bc_id, vlm_id)
        resolved.append(sub or part)
        if ncit:
            ncit_codes.append(ncit)
    return '; '.join(resolved), '; '.join(ncit_codes) if ncit_codes else None


print('Lookup functions defined (ct_lookup + ct_lookup_multi)')


## 6. Load BC and Dataset Specialization Data -- All Domains

No domain filtering -- load everything. The BC Hierarchy sheet provides
pre-computed hierarchy paths.

In [ ]:
BC_SHEET = 'Biomedical Concepts'
BC_HIER_SHEET = 'BC Hierarchy'
DS_SHEET = 'SDTM Dataset Specializations'

bc_df = pd.read_excel(BC_FILE, sheet_name=BC_SHEET)
bc_hier_df = pd.read_excel(BC_FILE, sheet_name=BC_HIER_SHEET)
ds_df = pd.read_excel(DS_FILE, sheet_name=DS_SHEET)

print(f'BC data elements: {len(bc_df)} rows, {bc_df["bc_id"].nunique()} unique BCs')
print(f'BC hierarchy:     {len(bc_hier_df)} rows (one per BC)')
print(f'DS rows:          {len(ds_df)} rows, {ds_df["vlm_group_id"].nunique()} unique DSS')
print(f'DS domains:       {sorted(ds_df["domain"].unique())}')


## 7. Build BC Index -- All BCs

One entry per bc_id. Uses BC Hierarchy sheet for pre-computed hierarchy.
No domain filtering -- every BC goes in.


In [ ]:
# Build hierarchy lookup from the BC Hierarchy sheet
hier_lookup = {}
for _, row in bc_hier_df.iterrows():
    bc_id = row.get('bc_id')
    if pd.isna(bc_id):
        continue
    hier_lookup[bc_id] = {
        'bc_hierarchy_level': int(row['bc_hierarchy_level']) if pd.notna(row.get('bc_hierarchy_level')) else 0,
        'bc_hierarchy_full': str(row['bc_hierarchy_full']) if pd.notna(row.get('bc_hierarchy_full')) else '',
        'dec_n': int(row['dec_n']) if pd.notna(row.get('dec_n')) else 0,
    }

# Build BC-level index (one row per bc_id from the data elements sheet)
bc_index = {}
for _, row in bc_df.iterrows():
    bc_id = row.get('bc_id')
    if pd.isna(bc_id) or bc_id in bc_index:
        continue  # Take first row per bc_id (BC-level fields are repeated)

    # Get parent label
    parent_id = row.get('parent_bc_id')
    parent_label = None
    if pd.notna(parent_id):
        parent_rows = bc_df[bc_df['bc_id'] == parent_id]
        if len(parent_rows) > 0:
            parent_label = parent_rows.iloc[0]['short_name']

    # Parse hierarchy path from pre-computed full path
    hier_info = hier_lookup.get(bc_id, {})
    hier_full = hier_info.get('bc_hierarchy_full', '')
    hierarchy_path = ''
    if hier_full:
        parts = [p.strip() for p in hier_full.split(';') if p.strip()]
        if len(parts) > 1:
            ancestors = parts[:-1]
            path_names = [re.sub(r'\s*\(C\w+\)\s*$', '', p).strip() for p in ancestors]
            hierarchy_path = ' > '.join(path_names)

    # Parse categories
    raw_cats = row.get('bc_categories', '')
    categories = ''
    if pd.notna(raw_cats) and raw_cats:
        cats = [c.strip() for c in str(raw_cats).split(';') if c.strip()]
        categories = '; '.join(sorted(cats))

    # Definition -- for LLM disambiguation (parallel to NCIt_Definition in SDTM_Test_Identity.xlsx)
    raw_def = row.get('definition', '')
    definition = str(raw_def).strip() if pd.notna(raw_def) and raw_def else ''

    # Synonyms
    raw_syn = row.get('synonyms', '')
    synonyms = str(raw_syn).strip() if pd.notna(raw_syn) and raw_syn else ''

    bc_index[bc_id] = {
        'bc_id': bc_id,
        'short_name': str(row.get('short_name', '')) if pd.notna(row.get('short_name')) else '',
        'ncit_code': str(row.get('ncit_code', '')) if pd.notna(row.get('ncit_code')) else '',
        'parent_bc_id': parent_id if pd.notna(parent_id) else None,
        'parent_label': parent_label,
        'bc_categories': categories,
        'synonyms': synonyms,
        'definition': definition,
        'result_scales': str(row.get('result_scales', '')) if pd.notna(row.get('result_scales')) else '',
        'hierarchy_path': hierarchy_path,
        'hierarchy_level': hier_info.get('bc_hierarchy_level', 0),
        'dec_n': hier_info.get('dec_n', 0),
    }

print(f'Built index for {len(bc_index)} BCs')
print(f'  With parent:     {sum(1 for v in bc_index.values() if v["parent_label"])}')
print(f'  Top-level:       {sum(1 for v in bc_index.values() if not v["parent_label"])}')
print(f'  With hierarchy:  {sum(1 for v in bc_index.values() if v["hierarchy_path"])}')
print(f'  With definition: {sum(1 for v in bc_index.values() if v["definition"])}')
print(f'  With synonyms:   {sum(1 for v in bc_index.values() if v["synonyms"])}')


## 8. Extract DS Data -- Domain-Generic

Instead of hardcoded LB variable names, extracts dimensions generically:
- **Topic**: identified by `role='Topic'` -> testCode, testName
- **Specimen**: variable ending in `SPEC` with assigned_term or value_list
- **Method**: variable ending in `METHOD` with assigned_term or value_list
- **Units**: variable ending in `ORRESU` (value_list) and `STRESU` (assigned_value)
- **LOINC**: variable ending in `LOINC`
- **Result**: variable ending in `ORRES` (data_type for Qn/Ql classification)
- **Category**: variable ending in `CAT` with assigned_value

The `role` column is authoritative for topic identification.
Variable suffixes handle the rest across all SDTM domains.

**Value list handling:** Specimen, method, and unit fields may contain
semicolon-separated value_lists (e.g. IS domain specimen: "SERUM; SERUM OR PLASMA; BLOOD").
All three use split-then-lookup-each via `ct_lookup_multi`, resolving each value
individually against SDTM CT before rejoining.


In [ ]:
def extract_ds_data(ds_rows, bc_id, domain):
    """Extract DS data from DS rows for a single vlm_group_id.
    Domain-generic: uses role column and SDTM suffix patterns."""
    ds_entry = {
        'vlm_group_id': None,
        'bc_id': bc_id,
        'domain': domain,
        'short_name': None,
        'specimen': None,
        'specimen_ncit': None,
        'method': None,
        'method_ncit': None,
        'units': [],
        'unit_details': [],
        'standardized_unit': None,
        'loinc_codes': [],
        'test_code': None,
        'test_name': None,
        'test_ncit': None,
        'result_type': None,
        'category': None,
        'subcategory': None,
        'location': None,
        'laterality': None,
        'evaluator': None,
    }

    vlm_id = None
    for _, row in ds_rows.iterrows():
        if pd.notna(row.get('vlm_group_id')):
            vlm_id = row['vlm_group_id']
            ds_entry['vlm_group_id'] = vlm_id
        if pd.notna(row.get('short_name')):
            ds_entry['short_name'] = row['short_name']

        sdtm_var = str(row.get('sdtm_variable', ''))
        role = str(row.get('role', ''))
        assigned_val = row.get('assigned_value')
        assigned_term = row.get('assigned_term')
        value_list = row.get('value_list')
        data_type = row.get('data_type')

        # Topic variable (role-based, domain-generic)
        if role == 'Topic':
            if pd.notna(assigned_val) and not ds_entry['test_code']:
                ds_entry['test_code'] = str(assigned_val).strip()
            if pd.notna(assigned_term) and not ds_entry['test_ncit']:
                ds_entry['test_ncit'] = str(assigned_term).strip()

        # Test name: --TEST variable (the Qualifier paired with --TESTCD)
        if sdtm_var.endswith('TEST') and not sdtm_var.endswith('TESTCD'):
            if pd.notna(assigned_val) and not ds_entry['test_name']:
                ds_entry['test_name'] = str(assigned_val).strip()

        # Specimen: --SPEC (assigned_term = single value, value_list = multi-value)
        if sdtm_var.endswith('SPEC'):
            spec_value = None
            is_value_list = False
            if pd.notna(assigned_term):
                spec_value = str(assigned_term)
            elif pd.notna(value_list):
                spec_value = str(value_list)
                is_value_list = True
            if spec_value:
                # Use ct_lookup_multi for value_lists (splits on semicolons)
                if is_value_list:
                    sub, ncit = ct_lookup_multi(spec_value, SPECIMEN_INDEX,
                                               MAPPED_SPECIMENS, UNMAPPED_SPECIMENS, bc_id, vlm_id)
                else:
                    sub, ncit = ct_lookup(spec_value, SPECIMEN_INDEX,
                                         MAPPED_SPECIMENS, UNMAPPED_SPECIMENS, bc_id, vlm_id)
                ds_entry['specimen'] = sub or spec_value
                ds_entry['specimen_ncit'] = ncit

        # Method: --METHOD (assigned_term = single value, value_list = multi-value)
        if sdtm_var.endswith('METHOD'):
            if pd.notna(assigned_term):
                sub, ncit = ct_lookup(assigned_term, METHOD_INDEX,
                                      MAPPED_METHODS, UNMAPPED_METHODS, bc_id, vlm_id)
                ds_entry['method'] = sub or str(assigned_term)
                ds_entry['method_ncit'] = ncit
            elif pd.notna(value_list):
                # value_list methods: split, look up each, rejoin
                sub, ncit = ct_lookup_multi(value_list, METHOD_INDEX,
                                            MAPPED_METHODS, UNMAPPED_METHODS, bc_id, vlm_id)
                ds_entry['method'] = sub or str(value_list)
                ds_entry['method_ncit'] = ncit

        # Units: --ORRESU (allowed units as value_list)
        if sdtm_var.endswith('ORRESU'):
            source = value_list if pd.notna(value_list) else assigned_val
            if pd.notna(source):
                units = [u.strip() for u in str(source).split(';') if u.strip()]
                for u in units:
                    sub, ncit = ct_lookup(u, UNIT_INDEX,
                                          MAPPED_UNITS, UNMAPPED_UNITS, bc_id, vlm_id)
                    ds_entry['units'].append(sub or u)
                    if sub:
                        ds_entry['unit_details'].append((sub, ncit))

        # Standardized unit: --STRESU
        if sdtm_var.endswith('STRESU'):
            if pd.notna(assigned_val):
                ds_entry['standardized_unit'] = str(assigned_val)

        # LOINC codes: --LOINC
        if sdtm_var.endswith('LOINC'):
            loinc_source = assigned_val if pd.notna(assigned_val) else value_list
            if pd.notna(loinc_source):
                loincs = [l.strip() for l in re.split(r'[;,]', str(loinc_source)) if l.strip()]
                ds_entry['loinc_codes'].extend(loincs)

        # Result type: --ORRES data_type for Qn/Ql
        if sdtm_var.endswith('ORRES') and pd.notna(data_type):
            ds_entry['result_type'] = str(data_type)

        # Category: --CAT
        if sdtm_var.endswith('CAT') and not sdtm_var.endswith('SCAT'):
            if pd.notna(assigned_val):
                ds_entry['category'] = str(assigned_val)

        # Subcategory: --SCAT
        if sdtm_var.endswith('SCAT'):
            if pd.notna(assigned_val):
                ds_entry['subcategory'] = str(assigned_val)

        # Location: --LOC
        if sdtm_var.endswith('LOC'):
            loc_source = assigned_val if pd.notna(assigned_val) else value_list
            if pd.notna(loc_source):
                ds_entry['location'] = str(loc_source)

        # Laterality: --LAT
        if sdtm_var.endswith('LAT'):
            if pd.notna(value_list):
                ds_entry['laterality'] = str(value_list)

        # Evaluator: --EVAL
        if sdtm_var.endswith('EVAL') and not sdtm_var.endswith('EVALIDX'):
            eval_source = assigned_val if pd.notna(assigned_val) else value_list
            if pd.notna(eval_source):
                ds_entry['evaluator'] = str(eval_source)

    # Deduplicate
    ds_entry['units'] = list(dict.fromkeys(ds_entry['units']))
    ds_entry['loinc_codes'] = list(dict.fromkeys(ds_entry['loinc_codes']))
    return ds_entry

# Extract all DS across all domains
ds_by_bc = defaultdict(list)
domain_ds_counts = Counter()

for vlm_id, group in ds_df.groupby('vlm_group_id'):
    if pd.isna(vlm_id):
        continue
    bc_id = group['bc_id'].iloc[0] if 'bc_id' in group.columns else None
    domain = group['domain'].iloc[0] if 'domain' in group.columns else '?'
    ds_data = extract_ds_data(group, bc_id, domain)
    if bc_id:
        ds_by_bc[bc_id].append(ds_data)
        domain_ds_counts[domain] += 1

print(f'Extracted DS for {len(ds_by_bc)} BCs')
total_ds = sum(len(v) for v in ds_by_bc.values())
print(f'Total DS: {total_ds}')
print(f'\nDS per domain:')
for domain, count in sorted(domain_ds_counts.items()):
    print(f'  {domain:5} {count:>5} DS')
print(f'\nCT mapping quality:')
print(f'  Specimens: {len(MAPPED_SPECIMENS)} mapped, {len(UNMAPPED_SPECIMENS)} unmapped')
print(f'  Methods:   {len(MAPPED_METHODS)} mapped, {len(UNMAPPED_METHODS)} unmapped')
print(f'  Units:     {len(MAPPED_UNITS)} mapped, {len(UNMAPPED_UNITS)} unmapped')


## 9. Architectural Note -- Categories vs Hierarchy

The BC export contains two independent classification axes:

| Source field | Representation | Purpose |
|---|---|---|
| `parent_bc_id` | Hierarchy path (pre-computed in BC Hierarchy sheet) | Structural BC-to-BC parent chain. Single-valued. |
| `bc_categories` | Category tags (semicolon-separated) | Curated NCI-EVS editorial tags for search/grouping. Multi-valued. |

These are **not** redundant -- categories serve discovery, hierarchy serves structure.
Both survive into the output independently as separate columns.


## 10. BC Type Classification

Now that we have all domains, the classification becomes accurate:

| BC_Type | Meaning |
|---|---|
| `full` | BC has Dataset Specializations with topic variable assigned |
| `full_no_ds` | BC defined but no DS in any domain |
| `hierarchy_only` | BC referenced as parent by other BCs, no own DS |

The LB-only version had 80 `full_no_ds` BCs -- 47 were hierarchy nodes,
16 had DS in non-LB domains. The all-domain version resolves both.


In [ ]:
# Identify which bc_ids are referenced as parents
all_parent_ids = set()
for info in bc_index.values():
    pid = info.get('parent_bc_id')
    if pid:
        all_parent_ids.add(pid)

# Classify each BC
ds_bc_ids = set(ds_by_bc.keys())

for bc_id, info in bc_index.items():
    has_ds = bc_id in ds_bc_ids
    is_parent = bc_id in all_parent_ids

    if has_ds:
        info['bc_type'] = 'full'
    elif is_parent:
        info['bc_type'] = 'hierarchy_only'
    else:
        info['bc_type'] = 'full_no_ds'

type_counts = Counter(info['bc_type'] for info in bc_index.values())
print('BC Type distribution:')
for bt, count in sorted(type_counts.items()):
    print(f'  {bt:20} {count:>5}')
print(f'  {"TOTAL":20} {len(bc_index):>5}')

# Which domains does each BC have DS in?
for bc_id, info in bc_index.items():
    ds_list = ds_by_bc.get(bc_id, [])
    domains = sorted(set(v['domain'] for v in ds_list if v.get('domain')))
    info['domains_with_ds'] = '; '.join(domains)
    domain_classes = sorted(set(get_domain_class(d) for d in domains))
    info['domain_class'] = '; '.join(domain_classes)

dc_counts = Counter()
for info in bc_index.values():
    dc = info.get('domain_class', '')
    if dc:
        for cls in dc.split('; '):
            dc_counts[cls] += 1
print(f'\nBC Domain Class distribution (BCs with DS):')
for cls, count in sorted(dc_counts.items()):
    print(f'  {cls:20} {count:>5} BCs')


## 11. Build Flat BC_DSS DataFrame

Single flat structure. One row per DSS or per BC-without-DSS.
Column order: BC identity → CT submission → classification → row structure
→ DSS dimensions → measurement spec → reference codes → COSMoS provenance.

**Row_Type values:**
- `DSS` -- one per Dataset Specialization
- `BC_only_no_DS` -- BC defined but no DSS exists
- `BC_only_hierarchy` -- structural parent node, no own DSS

In [ ]:
def resolve_result_scale(result_type, units, standardized_unit, bc_result_scales):
    """Resolve Result_Scale from DS-level data_type, with BC-level fallback."""
    if result_type:
        rt = result_type.upper()
        if 'INTEGER' in rt or 'FLOAT' in rt or 'DECIMAL' in rt:
            return 'Quantitative'
        elif 'TEXT' in rt or 'CHAR' in rt:
            return 'Qualitative'
        else:
            return result_type
    if units or standardized_unit:
        return 'Quantitative'
    if bc_result_scales:
        return bc_result_scales
    return ''


bc_dss_data = []

for bc_id, info in bc_index.items():
    bc_ds_list = ds_by_bc.get(bc_id, [])
    bc_type = info.get('bc_type', 'full_no_ds')

    # Map bc_type to Row_Type vocabulary
    if bc_type == 'hierarchy_only':
        row_type_bc = 'BC_only_hierarchy'
    elif bc_type == 'full_no_ds':
        row_type_bc = 'BC_only_no_DS'
    else:
        row_type_bc = 'DSS'  # overridden per-DSS below

    # Derive BC_Scope: Subject (patient-level) vs Study (trial-level metadata)
    bc_categories = info.get('bc_categories', '')
    is_study_level = (
        'Trial Summary' in bc_categories
        or 'Clinical Trial Attribute' in bc_categories
    )
    bc_scope = 'Study' if is_study_level else 'Subject'

    # Shared BC-level fields (same on every row for this BC)
    bc_fields = {
        'COSMoS_BC_ID':         bc_id,
        'NCIt_Code':             info.get('ncit_code', ''),
        'BC_Name':               info.get('short_name', ''),
        'BC_Definition':         info.get('definition', ''),
        'BC_Synonyms':           info.get('synonyms', ''),
        'BC_Type':               bc_type,
        'BC_Scope':              bc_scope,
        'TESTCD':                '',
        'TEST':                  '',
        'TESTCD_NCIt':           '',
        'Categories':            info.get('bc_categories', ''),
        'Hierarchy_Path':        info.get('hierarchy_path', ''),
        'Parent_Label':          info.get('parent_label') or '',
        'Domains_with_DS': info.get('domains_with_ds', ''),
        'Decimal_Places':        info.get('dec_n', 0) or '',
    }

    if bc_ds_list:
        for v in bc_ds_list:
            result_scale = resolve_result_scale(
                v['result_type'], v['units'], v['standardized_unit'],
                info.get('result_scales', '')
            )
            row = dict(bc_fields)
            row.update({
                'Row_Type':           'DSS',
                'TESTCD':             v['test_code'] or '',
                'TEST':              v['test_name'] or '',
                'TESTCD_NCIt':       v['test_ncit'] or '',
                'DS_Code':              v['vlm_group_id'] or '',
                'DS_Name':       v['short_name'] or '',
                'Domain':             v['domain'] or '',
                'Domain_Class':       get_domain_class(v['domain']),
                'Result_Scale':       result_scale,
                'Specimen':           v['specimen'] or '',
                'Specimen_NCIt':      v['specimen_ncit'] or '',
                'Method':             v['method'] or '',
                'Method_NCIt':        v['method_ncit'] or '',
                'LOINC_Code_Count':    str(len(v['loinc_codes'])),
                'LOINC_Code':         '; '.join(v['loinc_codes']),
                'Allowed_Units':      '; '.join(v['units']),
                'Standard_Unit':      v['standardized_unit'] or '',
                'COSMoS_Category':    v['category'] or '',
                'COSMoS_Subcategory': v['subcategory'] or '',
                'location':           v['location'] or '',
                'laterality':         v['laterality'] or '',
                'evaluator':          v['evaluator'] or '',
            })
            bc_dss_data.append(row)
    else:
        # BC without DS -- one row, measurement columns blank
        row = dict(bc_fields)
        row.update({
            'Row_Type':           row_type_bc,
            'DS_Code':              '',
            'DS_Name':       '',
            'Domain':             '',
            'Domain_Class':       info.get('domain_class', ''),
            'Result_Scale':       info.get('result_scales', ''),
            'Specimen':           '',
            'Specimen_NCIt':      '',
            'Method':             '',
            'Method_NCIt':        '',
            'LOINC_Code_Count':    '',
            'LOINC_Code':         '',
            'Allowed_Units':      '',
            'Standard_Unit':      '',
            'COSMoS_Category':    '',
            'COSMoS_Subcategory': '',
            'location':           '',
            'laterality':         '',
            'evaluator':          '',
        })
        bc_dss_data.append(row)


# Authoritative column order
COLUMN_ORDER = [
    # BC identity
    'COSMoS_BC_ID', 'NCIt_Code', 'BC_Name', 'BC_Definition', 'BC_Synonyms', 'BC_Type', 'BC_Scope',
    # CT submission
    'TESTCD', 'TEST', 'TESTCD_NCIt',
    # Classification
    'Categories', 'Hierarchy_Path', 'Parent_Label',
    # Row structure
    'Row_Type', 'Domains_with_DS',
    # DS identity
    'DS_Code', 'DS_Name', 'Domain', 'Domain_Class',
    # Measurement specification
    'Result_Scale', 'Specimen', 'Specimen_NCIt', 'Method', 'Method_NCIt',
    # Reference codes
    'LOINC_Code_Count', 'LOINC_Code', 'Allowed_Units', 'Standard_Unit',
    # COSMoS provenance
    'COSMoS_Category', 'COSMoS_Subcategory', 'Decimal_Places',
    # Contextual qualifiers (sparse -- domain-specific)
    'location', 'laterality', 'evaluator',
]

bc_dss_df = pd.DataFrame(bc_dss_data)[COLUMN_ORDER]

n_dss_rows   = (bc_dss_df['Row_Type'] == 'DSS').sum()
n_no_ds_rows = (bc_dss_df['Row_Type'] == 'BC_only_no_DS').sum()
n_hier_rows  = (bc_dss_df['Row_Type'] == 'BC_only_hierarchy').sum()
dss_rows     = bc_dss_df[bc_dss_df['Row_Type'] == 'DSS'].copy()

print(f'BC_DSS: {len(bc_dss_df)} rows x {len(COLUMN_ORDER)} columns')
print(f'  Row_Type=DSS:              {n_dss_rows}')
print(f'  Row_Type=BC_only_no_DS:    {n_no_ds_rows}')
print(f'  Row_Type=BC_only_hierarchy:{n_hier_rows}')
print(f'\nColumn population ({len(COLUMN_ORDER)} columns):')
for c in COLUMN_ORDER:
    n = (bc_dss_df[c].astype(str).str.strip().str.len() > 0).sum()
    print(f'  {c:25} {n:5}/{len(bc_dss_df)} populated')


## 12. Generate Statistics


In [ ]:
stats = {
    'Total BCs':                   len(bc_index),
    'BCs with DSS (full)':    int(bc_dss_df[bc_dss_df['BC_Type'] == 'full']['COSMoS_BC_ID'].nunique()),
    'BCs without DSS (full_no_ds)': int(bc_dss_df[bc_dss_df['Row_Type'] == 'BC_only_no_DS']['COSMoS_BC_ID'].nunique()),
    'Hierarchy-only BCs':          int(bc_dss_df[bc_dss_df['Row_Type'] == 'BC_only_hierarchy']['COSMoS_BC_ID'].nunique()),
    'Total Dataset Specializations':int(n_dss_rows),
    'SDTM domains with DSS':   int(dss_rows['Domain'].nunique()),
    'DSS with specimen':       int((dss_rows['Specimen'].str.len() > 0).sum()),
    'DSS with LOINC':          int((dss_rows['LOINC_Code'].str.len() > 0).sum()),
    'DSS with standard unit':  int((dss_rows['Standard_Unit'].str.len() > 0).sum()),
    'DSS Quantitative':        int((dss_rows['Result_Scale'] == 'Quantitative').sum()),
    'DSS Qualitative':         int((dss_rows['Result_Scale'] == 'Qualitative').sum()),
}

hierarchy_stats = {
    'BCs with parent': int((bc_dss_df.drop_duplicates('COSMoS_BC_ID')['Parent_Label'].str.len() > 0).sum()),
    'Top-level BCs':   int((bc_dss_df.drop_duplicates('COSMoS_BC_ID')['Parent_Label'].str.len() == 0).sum()),
}

all_category_tags = []
for cats_str in bc_dss_df.drop_duplicates('COSMoS_BC_ID')['Categories']:
    if cats_str:
        for tag in str(cats_str).split(';'):
            tag = tag.strip()
            if tag:
                all_category_tags.append(tag)
category_tag_counts = Counter(all_category_tags)

domain_class_bc_stats = {}
for cls in sorted(set(dss_rows['Domain_Class'])):
    cls_domains = dss_rows[dss_rows['Domain_Class'] == cls]['Domain'].unique()
    cls_bcs = dss_rows[dss_rows['Domain_Class'] == cls]['COSMoS_BC_ID'].nunique()
    cls_ds = int((dss_rows['Domain_Class'] == cls).sum())
    domain_class_bc_stats[cls] = {'domains': len(cls_domains), 'bcs': cls_bcs, 'ds_count': cls_ds}

domain_summary_data = []
for domain in sorted(dss_rows['Domain'].unique()):
    dom_v = dss_rows[dss_rows['Domain'] == domain]
    domain_summary_data.append({
        'Domain':        domain,
        'Domain_Label':  get_domain_label(domain),
        'Domain_Class':  get_domain_class(domain),
        'BC_Count':      dom_v['COSMoS_BC_ID'].nunique(),
        'DS_Count': len(dom_v),
        'With_Specimen': int((dom_v['Specimen'].str.len() > 0).sum()),
        'With_Method':   int((dom_v['Method'].str.len() > 0).sum()),
        'With_Unit':     int((dom_v['Standard_Unit'].str.len() > 0).sum()),
        'With_LOINC':    int((dom_v['LOINC_Code'].str.len() > 0).sum()),
        'Quantitative':  int((dom_v['Result_Scale'] == 'Quantitative').sum()),
        'Qualitative':   int((dom_v['Result_Scale'] == 'Qualitative').sum()),
    })
domain_summary_df = pd.DataFrame(domain_summary_data)

print('=' * 70)
print('STATISTICS SUMMARY')
print('=' * 70)
for key, value in stats.items():
    print(f'{key:40} {value:>8}')
print()
for key, value in hierarchy_stats.items():
    print(f'{key:40} {value:>8}')
print(f'  Unique category tags:                  {len(category_tag_counts):>8}')
print(f'\nDomain class distribution:')
for cls, info in sorted(domain_class_bc_stats.items()):
    print(f'  {cls:20} {info["domains"]:>3} domains, {info["bcs"]:>5} BCs, {info["ds_count"]:>5} DSS')
print(f'\nTop 15 category tags:')
for tag, count in category_tag_counts.most_common(15):
    print(f'  {tag:50} {count:>5}')
print(f'\nDomain summary:')
print(domain_summary_df.to_string(index=False))


## 13. Serialize CT Unmapped Log

Serializes CT unmapped trackers built during DS extraction into a structured sheet.

**Purpose:** Allows `COSMoS_BC_DSS_Validate.ipynb` to reconstruct these
trackers from the interim file without re-downloading and re-processing sources.
Keeps Validate independent of source files.

**Columns:** `Type` (SPECIMEN | METHOD | UNIT) | `Value` (unresolved string) | `Usage_Count`

In [ ]:
unmapped_log_data = []

for val, usages in sorted(UNMAPPED_SPECIMENS.items()):
    unmapped_log_data.append({'Type': 'SPECIMEN', 'Value': val, 'Usage_Count': len(usages)})

for val, usages in sorted(UNMAPPED_METHODS.items()):
    unmapped_log_data.append({'Type': 'METHOD', 'Value': val, 'Usage_Count': len(usages)})

for val, usages in sorted(UNMAPPED_UNITS.items()):
    unmapped_log_data.append({'Type': 'UNIT', 'Value': val, 'Usage_Count': len(usages)})

ct_unmapped_df = pd.DataFrame(unmapped_log_data) if unmapped_log_data else pd.DataFrame(
    columns=['Type', 'Value', 'Usage_Count']
)

print(f'CT_Unmapped_Log: {len(ct_unmapped_df)} entries')
print(f'  SPECIMEN: {len(ct_unmapped_df[ct_unmapped_df["Type"] == "SPECIMEN"])}')
print(f'  METHOD:   {len(ct_unmapped_df[ct_unmapped_df["Type"] == "METHOD"])}')
print(f'  UNIT:     {len(ct_unmapped_df[ct_unmapped_df["Type"] == "UNIT"])}')

## 14. Export to Excel (Interim)

In [ ]:
excel_output = INTERIM_DIR / 'COSMoS_BC_DSS.xlsx'

# --- Write data sheets first via pandas ---
with pd.ExcelWriter(excel_output, engine='openpyxl') as writer:
    bc_dss_df.to_excel(writer, sheet_name='BC_DSS', index=False)
    domain_summary_df.to_excel(writer, sheet_name='Domain_Summary', index=False)

    stats_rows = [['Metric', 'Value', 'Detail'], ['', '', ''], ['OVERVIEW', '', '']]
    for k, v in stats.items():
        stats_rows.append([k, v, ''])
    stats_rows += [['', '', ''], ['HIERARCHY', '', '']]
    for k, v in hierarchy_stats.items():
        stats_rows.append([k, v, ''])
    stats_rows.append(['Unique category tags', len(category_tag_counts), ''])
    stats_rows += [['', '', ''], ['DOMAIN CLASS DISTRIBUTION', '', '']]
    for cls, info in sorted(domain_class_bc_stats.items()):
        stats_rows.append([cls, info['bcs'], f'{info["domains"]} domains, {info["ds_count"]} DSS'])
    stats_rows += [['', '', ''], ['TOP 20 CATEGORY TAGS', 'Count', '']]
    for tag, count in category_tag_counts.most_common(20):
        stats_rows.append([tag, count, ''])
    stats_df = pd.DataFrame(stats_rows[1:], columns=stats_rows[0])
    stats_df.to_excel(writer, sheet_name='Statistics', index=False)

    ct_unmapped_df.to_excel(writer, sheet_name='CT_Unmapped_Log', index=False)

# --- Add styled README sheet (single-column, aligned with SDTM_Test_Identity.xlsx pattern) ---
wb = load_workbook(excel_output)

readme_title   = Font(name='Arial', bold=True, size=14, color='B8860B')
readme_section = Font(name='Arial', bold=True, size=11, color='B8860B')
readme_body    = Font(name='Arial', size=10)
readme_bold    = Font(name='Arial', bold=True, size=10)

# Domain legend for COLUMN DESCRIPTIONS
domain_legend_lines = []
for domain in sorted(dss_rows['Domain'].unique()):
    label = get_domain_label(domain)
    cls = get_domain_class(domain)
    domain_legend_lines.append((f'    {domain:5} = {label} ({cls})', readme_body))

readme_lines = [
    ('COSMoS BC + DSS — FLAT INTERIM REFERENCE FILE', readme_title),
    ('', None),

    ('PROVENANCE', readme_section),
    ('', None),
    (f'Generated:       {GENERATION_DATE}', readme_body),
    (f'Pipeline:        COSMoS_BC_DSS_Flatten → COSMoS_BC_DSS_Validate', readme_body),
    (f'BC source:       {BC_FILE.name} (CDISC COSMoS GitHub)', readme_body),
    (f'DS source:       {DS_FILE.name} (CDISC COSMoS GitHub)', readme_body),
    (f'SDTM CT source:  {SDTM_CT_FILE.name} (NCI EVS, for specimen/method/unit lookup)', readme_body),
    ('', None),

    ('SCOPE', readme_section),
    ('', None),
    (f'All {len(bc_index)} BCs across {dss_rows["Domain"].nunique()} SDTM domains.', readme_body),
    ('One row per Dataset Specialization (DSS). A BC with multiple specimen/method', readme_body),
    ('variants produces multiple rows. BCs without DSS are included as gap markers.', readme_body),
    ('', None),
    (f'  DSS rows:                    {n_dss_rows:,}', readme_body),
    (f'  BCs without DSS:             {n_no_ds_rows:,}', readme_body),
    (f'  Hierarchy-only BCs:          {n_hier_rows:,}', readme_body),
    ('', None),

    ('SHEETS', readme_section),
    ('', None),
    ('README             This sheet', readme_body),
    ('BC_DSS             Primary data — one row per DSS or BC-without-DSS', readme_body),
    ('Domain_Summary     Per-domain statistics with observation class', readme_body),
    ('Statistics         Overall metrics, domain class distribution, category tags', readme_body),
    ('CT_Unmapped_Log    CT lookup failures — consumed by Validate for QC-01/02/03', readme_body),
    ('', None),

    ('COLUMN DESCRIPTIONS', readme_section),
    ('', None),
    ('COSMoS_BC_ID       COSMoS internal BC identifier (bc_id). May be a placeholder (NEW_*).', readme_body),
    ('NCIt_Code          NCIt C-code. Blank = no NCIt assignment yet. Join key to SDTM_Test_Identity.xlsx.', readme_body),
    ('BC_Name            BC preferred name (COSMoS short_name). Primary semantic signal.', readme_body),
    ('BC_Definition      Plain-language definition. Coverage varies. Use for disambiguation.', readme_body),
    ('BC_Synonyms        Alternative names (semicolon-separated). Use for name matching.', readme_body),
    ('BC_Type            full | full_no_ds | hierarchy_only — classification of the BC itself.', readme_body),
    ('BC_Scope           Subject | Study. Subject = patient-level observation. Study = trial-level', readme_body),
    ('                   metadata (TS parameters, Clinical Trial Attributes). Derived from bc_categories.', readme_body),
    ('TESTCD             CDISC submission value (8 chars max). Join key to SDTM_Test_Identity.xlsx.', readme_body),
    ('TEST               CDISC full test name (submission value).', readme_body),
    ('TESTCD_NCIt        NCIt code for the TESTCD (from DSS assigned_term). Usually equals NCIt_Code.', readme_body),
    ('                   Differs for ~6 tests where the BC and TESTCD carry different NCIt codes. Reason unknown.', readme_body),
    ('Categories         Editorial category tags (semicolon-separated, multi-valued). From bc_categories.', readme_body),
    ('Hierarchy_Path     Structural ancestor chain (grandparent > parent). From parent_bc_id. Excludes self.', readme_body),
    ('Parent_Label       Immediate parent BC name. Blank for top-level BCs.', readme_body),
    ('Row_Type           DSS | BC_only_no_DS | BC_only_hierarchy — classification of this row.', readme_body),
    ('Domains_with_DS    SDTM domains where this BC has DSS (semicolon-separated).', readme_body),
    ('DS_Code            Dataset Specialization code (COSMoS vlm_group_id). Blank for BC_only rows.', readme_body),
    ('DS_Name            DS descriptive name (e.g. Albumin in Serum or Plasma).', readme_body),
    ('Domain             SDTM domain code (e.g. LB, EG, VS). Blank for BC_only rows.', readme_body),
    ('Domain_Class       SDTM observation class: Findings | Events | Interventions | Special-Purpose.', readme_body),
    ('                   Source: hardcoded from SDTM model (not publicly downloadable as open data).', readme_body),
    ('Result_Scale       Quantitative | Qualitative | datetime. Derived from ORRES data_type.', readme_body),
    ('Specimen           CDISC CT submission value for specimen type (e.g. SERUM OR PLASMA).', readme_body),
    ('Specimen_NCIt      NCIt C-code for specimen.', readme_body),
    ('Method             CDISC CT submission value for method. Blank if not specified.', readme_body),
    ('Method_NCIt        NCIt C-code for method.', readme_body),
    ('LOINC_Code_Count   Number of LOINC codes for this DS. Count > 1 = unit-convention variants.', readme_body),
    ('LOINC_Code         LOINC codes (semicolon-separated). Primarily LB and MB domains.', readme_body),
    ('Allowed_Units      Allowed result units — ORRESU value_list (semicolon-separated).', readme_body),
    ('Standard_Unit      Standardized reporting unit — STRESU assigned_value.', readme_body),
    ('COSMoS_Category    DS-level --CAT value. Different system from BC-level Categories.', readme_body),
    ('COSMoS_Subcategory DS-level --SCAT value.', readme_body),
    ('Decimal_Places     Recommended decimal precision (dec_n from BC Hierarchy sheet).', readme_body),
    ('location           --LOC value. Sparse: VS, EC, MK, and a few other domains.', readme_body),
    ('laterality         --LAT value_list. Sparse: VS, EC, RP, UR.', readme_body),
    ('evaluator          --EVAL value. Sparse: EG, FA, TU, RS, QS.', readme_body),
    ('', None),
    ('Domain codes present:', readme_bold),
] + domain_legend_lines + [
    ('', None),

    ('COVERAGE STATISTICS', readme_section),
    ('', None),
    (f'Total BCs:                     {len(bc_index):,}', readme_bold),
    (f'  With DSS (full):             {int(bc_dss_df[bc_dss_df["BC_Type"] == "full"]["COSMoS_BC_ID"].nunique()):,}', readme_body),
    (f'  Without DSS (full_no_ds):    {n_no_ds_rows:,}', readme_body),
    (f'  Hierarchy-only:              {n_hier_rows:,}', readme_body),
    (f'Total DSS:                     {n_dss_rows:,}', readme_bold),
    (f'  With specimen:               {int((dss_rows["Specimen"].str.len() > 0).sum()):,}', readme_body),
    (f'  With LOINC:                  {int((dss_rows["LOINC_Code"].str.len() > 0).sum()):,}', readme_body),
    (f'  With standard unit:          {int((dss_rows["Standard_Unit"].str.len() > 0).sum()):,}', readme_body),
    (f'  Quantitative:                {int((dss_rows["Result_Scale"] == "Quantitative").sum()):,}', readme_body),
    (f'  Qualitative:                 {int((dss_rows["Result_Scale"] == "Qualitative").sum()):,}', readme_body),
    ('', None),

    ('DESIGN DECISIONS', readme_section),
    ('', None),
    ('Column naming is interim. COSMoS source field names are translated to more readable', readme_body),
    ('forms, but not yet finalized. The mapping from COSMoS fields to output columns is', readme_body),
    ('documented in the Flatten notebook (Cell 1). Final naming pending community discussion.', readme_body),
    ('', None),
    ('Domain_Class is hardcoded. The SDTM model metadata that defines domain observation', readme_body),
    ('classes is not published as open data (requires CDISC membership). The classification', readme_body),
    ('is verified against SDTM v2.0 model metadata. Unknown domains return "Unknown".', readme_body),
    ('', None),
    ('Categories and Hierarchy_Path are independent. Categories come from bc_categories', readme_body),
    ('(editorial tags for discovery). Hierarchy comes from parent_bc_id (structural', readme_body),
    ('parent chain). They are not redundant and are preserved as separate columns.', readme_body),
    ('', None),

    ('KNOWN LIMITATIONS', readme_section),
    ('', None),
    ('1. BC_Definition coverage varies — not all BCs have definitions in COSMoS source.', readme_body),
    ('2. BC_Synonyms are undifferentiated — no type annotation from COSMoS.', readme_body),
    ('3. LOINC codes present primarily in LB and MB domains only.', readme_body),
    ('4. LOINC display names not included — planned for enrichment step.', readme_body),
    ('5. Specimen equivalence: SERUM OR PLASMA accepts SERUM and PLASMA. Other equivalences need rules.', readme_body),
    ('6. NCIt_Code blanks: placeholder BCs (NEW_*) have no NCIt code — no join to SDTM_Test_Identity.xlsx.', readme_body),
    ('7. TESTCD_NCIt != NCIt_Code for ~6 tests (e.g. GLUCPE, HEIGHT, WEIGHT). The BC-level and TESTCD-level', readme_body),
    ('   NCIt codes differ. Possibly a legacy artefact from pre-COSMoS NCIt assignments, but the reason is', readme_body),
    ('   not confirmed. Both codes are valid; NCIt_Code identifies the BC, TESTCD_NCIt identifies the test.', readme_body),
    ('8. BC_Scope: COSMoS includes Trial Summary parameters and Clinical Trial Attributes as BCs,', readme_body),
    ('   although these are study-level metadata, not patient-level observations. Filter on', readme_body),
    ('   BC_Scope=Subject to exclude. See docs/COSMoS_Study_Design_Questions.md for discussion.', readme_body),
]

# Create README sheet and move it to first position
ws_readme = wb.create_sheet('README', 0)
ws_readme.sheet_properties.tabColor = 'B8860B'

for ri, (text, font) in enumerate(readme_lines, start=1):
    cell = ws_readme.cell(row=ri, column=1, value=text)
    if font:
        cell.font = font
ws_readme.column_dimensions['A'].width = 100

wb.save(excel_output)
print(f'[OK] Written to {excel_output.name}')



## 15. Format Excel

In [ ]:
wb = load_workbook(excel_output)

YELLOW_FILL       = PatternFill(start_color='FFFCE8', end_color='FFFCE8', fill_type='solid')
YELLOW_HEADER     = PatternFill(start_color='FFD700', end_color='FFD700', fill_type='solid')
README_HEADER     = PatternFill(start_color='595959', end_color='595959', fill_type='solid')
README_SECTION    = PatternFill(start_color='D9D9D9', end_color='D9D9D9', fill_type='solid')
SECTION_FILL      = PatternFill(start_color='FFE6CC', end_color='FFE6CC', fill_type='solid')
BCONLY_NODS_FILL  = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')  # amber -- gap
BCONLY_HIER_FILL  = PatternFill(start_color='EDEDED', end_color='EDEDED', fill_type='solid')  # grey -- structural
HEADER_FONT       = Font(bold=True)
HEADER_FONT_WHITE = Font(bold=True, color='FFFFFF')
thin_border = Border(
    left=Side(style='thin', color='999999'),
    right=Side(style='thin', color='999999'),
    top=Side(style='thin', color='999999'),
    bottom=Side(style='thin', color='999999'),
)

def format_sheet_basic(ws, header_fill, header_font, col_widths=None):
    """Apply basic formatting: borders, header style, column widths."""
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
        for cell in row:
            cell.border = thin_border
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
    if col_widths:
        for col_letter, width in col_widths.items():
            ws.column_dimensions[col_letter].width = width

# -- README (single-column, already styled during write -- no reformatting needed) --
print('README: styled during export (single-column)')

# -- BC_DSS --
ws = wb['BC_DSS']
col_names = [cell.value for cell in ws[1]]
row_type_col_idx = col_names.index('Row_Type') + 1 if 'Row_Type' in col_names else None

# Header row
for cell in ws[1]:
    cell.font = HEADER_FONT
    cell.fill = YELLOW_HEADER
    cell.border = thin_border
    cell.alignment = Alignment(wrap_text=True, vertical='top')

# Data rows -- fill by Row_Type
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    row_type_val = row[row_type_col_idx - 1].value if row_type_col_idx else ''
    if row_type_val == 'BC_only_no_DS':
        fill = BCONLY_NODS_FILL
    elif row_type_val == 'BC_only_hierarchy':
        fill = BCONLY_HIER_FILL
    else:
        fill = YELLOW_FILL
    for cell in row:
        cell.fill = fill
        cell.border = thin_border
        cell.alignment = Alignment(wrap_text=True, vertical='top')

# Column widths
col_widths_map = {
    'COSMoS_BC_ID': 14, 'NCIt_Code': 14, 'BC_Name': 40, 'BC_Definition': 55,
    'BC_Synonyms': 35, 'BC_Type': 16, 'TESTCD': 12, 'TEST': 38,
    'Categories': 50, 'Hierarchy_Path': 50, 'Parent_Label': 35,
    'Row_Type': 20, 'Domains_with_DS': 22,
    'DS_Code': 16, 'DS_Name': 45, 'Domain': 10, 'Domain_Class': 18,
    'Result_Scale': 14, 'Specimen': 22, 'Specimen_NCIt': 14,
    'Method': 30, 'Method_NCIt': 14,
    'LOINC_Code_Count': 14, 'LOINC_Code': 30, 'Allowed_Units': 28, 'Standard_Unit': 16,
    'COSMoS_Category': 20, 'COSMoS_Subcategory': 20, 'Decimal_Places': 14,
    'location': 18, 'laterality': 14, 'evaluator': 18,
}
for idx, col_name in enumerate(col_names, 1):
    letter = get_column_letter(idx)
    ws.column_dimensions[letter].width = col_widths_map.get(col_name, 14)

ws.auto_filter.ref = ws.dimensions
ws.freeze_panes = 'A2'
print('Formatted BC_DSS')

# -- Domain_Summary --
ws = wb['Domain_Summary']
format_sheet_basic(ws, YELLOW_HEADER, HEADER_FONT)
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for cell in row:
        cell.fill = YELLOW_FILL
ws.column_dimensions['A'].width = 10
ws.column_dimensions['B'].width = 30
ws.column_dimensions['C'].width = 18
for letter in ['D', 'E', 'F', 'G', 'H', 'I', 'J', 'K']:
    ws.column_dimensions[letter].width = 15
print('Formatted Domain_Summary')

# -- Statistics --
ws = wb['Statistics']
format_sheet_basic(ws, README_HEADER, HEADER_FONT_WHITE, {'A': 35, 'B': 20, 'C': 40})
for row in ws.iter_rows(min_row=2):
    val = str(row[0].value) if row[0].value else ''
    if val and val == val.upper() and val.strip():
        for cell in row:
            cell.font = HEADER_FONT
            cell.fill = SECTION_FILL
print('Formatted Statistics')

# -- CT_Unmapped_Log --
LOG_FILL = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
ws = wb['CT_Unmapped_Log']
format_sheet_basic(ws, README_HEADER, HEADER_FONT_WHITE, {'A': 12, 'B': 50, 'C': 14})
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for cell in row:
        cell.fill = LOG_FILL
print('Formatted CT_Unmapped_Log')
wb.save(excel_output)
print(f'\n[OK] Formatted Excel: {excel_output.name}')



## 16. Final Summary

In [ ]:
print('=' * 70)
print('COSMoS_BC_DSS_Flatten -- COMPLETE')
print('=' * 70)
print(f'\nScope: {len(bc_index)} BCs across {dss_rows["Domain"].nunique()} SDTM domains')
print(f'  Row_Type=DSS:              {n_dss_rows}')
print(f'  Row_Type=BC_only_no_DS:    {n_no_ds_rows}')
print(f'  Row_Type=BC_only_hierarchy:{n_hier_rows}')
print(f'\nDomain class distribution:')
for cls, info in sorted(domain_class_bc_stats.items()):
    print(f'  {cls:20} {info["domains"]:>3} domains, {info["bcs"]:>5} BCs, {info["ds_count"]:>5} DSS')
print(f'\nOutput: {excel_output}')
print(f'  README          -- provenance, scope, column descriptions, coverage')
print(f'  BC_DSS          -- {len(bc_dss_df)} rows x {len(COLUMN_ORDER)} columns (primary sheet)')
print(f'  Domain_Summary  -- per-domain statistics')
print(f'  Statistics      -- overall metrics and distribution')
print(f'  CT_Unmapped_Log -- {len(ct_unmapped_df)} unmapped CT values (consumed by Validate)')
print(f'\nNext step: run COSMoS_BC_DSS_Validate.ipynb')
print('=' * 70)

